# 01 p53 ODE Model Workflow

This notebook explains the professor's p53 ODE model workflow at a high level and shows a small toy example of the type of response curve we will use later.

The full professor model is **not implemented here yet**. The aim of this notebook is to make the workflow clear before adapting it to TCGA-BRCA breast cancer and doxorubicin.

## Why p53 Matters for Doxorubicin

Doxorubicin is a DNA-damaging chemotherapy. When DNA damage occurs, cells activate DNA-damage response pathways that can pause the cell cycle, repair damage, or trigger cell death if the damage is too severe.

p53 is a key tumour suppressor in this process. It helps decide whether a damaged cell should repair, stop dividing, or undergo apoptosis. Because of this, p53 signalling is biologically relevant when studying response or resistance to DNA-damaging drugs such as doxorubicin.

In this assignment, p53 is used as a mechanistic bridge between molecular data and treatment response. The question is not simply whether one gene is high or low, but whether a simulated p53 response pattern might help explain survival or drug sensitivity.

## What DDR Means in the Model

`DDR` means DNA-damage response input. In the professor's p53 model, `DDR` is used as a model input that represents the strength of DNA damage or DNA-damage signalling.

For the breast cancer and doxorubicin assignment, we can think of increasing `DDR` values as increasing DNA-damage stress. The original professor example simulates the model across a range of `DDR` doses, such as `DDR_0.010` through `DDR_1.000`.

This is a simplified modelling input. It is not a measured TCGA time-course variable.

## What p53 S15 and p53 S46 Represent

The professor model outputs simulated p53 phosphorylation response scores:

- **p53 S15**: simulated phosphorylation of p53 at serine 15. This is linked to DNA-damage signalling and p53 activation.
- **p53 S46**: simulated phosphorylation of p53 at serine 46. This is often interpreted as being closer to stronger stress or apoptotic p53 signalling.

The important point for this assignment is that these are **model-derived response scores**. They summarize how the p53 model responds to a given `DDR` input for a sample.

## How These Scores Could Be Used Later

Once we have TCGA-BRCA molecular data prepared, a later notebook can adapt the model inputs to breast cancer samples and produce simulated p53 response scores.

Those scores could then be tested against:

- patient survival in TCGA-BRCA;
- doxorubicin sensitivity in breast cancer cell lines;
- machine-learning or regression signatures built from expression data.

This makes the ODE model useful as a pre-specified mechanistic feature generator. It is different from training a model directly from TCGA time-course data. TCGA-BRCA does not provide doxorubicin time-course p53 measurements for each patient.

## Conceptual Workflow

The planned workflow is:

```text
DDR input -> p53 ODE model -> p53 S15/S46 simulated response scores -> survival or drug-response testing
```

JNK is excluded from this assignment plan and is not used in this notebook.

## What Was Found in the Professor Materials

The local professor files were available during this review. The original workflow is a neuroblastoma/TARGET example, not a TCGA-BRCA breast cancer workflow. We should treat it as a template to adapt, not as code to run unchanged.

The main p53 model files reviewed were:

- `p53_model_updated.ipynb`
- `p53_parameters_summaryTable.txt`
- `nb_params_full.csv`
- `p53s15DR_tp53.csv`
- `p53s46DR_tp53.csv`

## Professor Model Inputs and Outputs

The prepared neuroblastoma input table, `nb_params_full.csv`, contains patient/sample identifiers, survival fields, and model input genes.

Required columns observed in that file:

| Column group | Columns |
| --- | --- |
| Sample identifiers | `SAMPLE_ID`, `PATIENT_ID` |
| Clinical fields | `CANCER_TYPE_DETAILED`, `TMB_NONSYNONYMOUS`, `OS_MONTHS`, `OS_STATUS`, `SURV_STATUS` |
| p53 model genes | `ATM`, `CHEK2`, `HIPK2`, `MDM2`, `PPM1D`, `SIAH1`, `TP53`, `WSB1` |

The parameter table, `p53_parameters_summaryTable.txt`, contains fitted parameter sets for the ODE model. Important groups include doxorubicin and radiation ATM activation parameters, CHK2 parameters, p53 phosphorylation/dephosphorylation parameters, SIAH1/HIPK2-related parameters, MDM2/WIP1 expression parameters, and scaling factors for S15/S46.

The output files are:

- `p53s15DR_tp53.csv`: simulated p53 S15 response scores across DDR doses.
- `p53s46DR_tp53.csv`: simulated p53 S46 response scores across DDR doses.

Both output files contain DDR dose columns from `DDR_0.010` to `DDR_1.000`, plus `SAMPLE_ID` and `PATIENT_ID`.

## Adaptation Needed for TCGA-BRCA

For this assignment, we will need to replace the neuroblastoma/TARGET inputs with TCGA-BRCA breast cancer inputs.

The simple adaptation idea is:

1. Prepare TCGA-BRCA expression and clinical survival data.
2. Extract the same model input genes where available: `ATM`, `CHEK2`, `HIPK2`, `MDM2`, `PPM1D`, `SIAH1`, `TP53`, and `WSB1`.
3. Use the professor's p53 parameter table as a pre-specified mechanistic model input.
4. Simulate p53 S15 and S46 response scores across DDR doses.
5. Test those scores against survival or drug response in later notebooks.

This notebook does not download TCGA, DepMap, PRISM, GDSC, or LINCS data and does not run survival or drug-response models.

## Toy Demonstration Only

The next code cell creates a **toy** p53 response curve. It is not the professor's full ODE model and should not be interpreted biologically.

The purpose is only to illustrate the assignment workflow:

- increase a DNA-damage input called `DDR`;
- calculate two simple response scores;
- plot dose versus simulated response;
- save a conceptual figure for the report.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


# If the notebook is run from the repository root, use figures/.
# If it is run from notebooks/, use ../figures/.
figures_dir = Path("figures")
if not figures_dir.exists():
    figures_dir = Path("../figures")

figures_dir.mkdir(exist_ok=True)

# Toy DDR doses matching the idea of the professor output columns.
ddr_doses = np.linspace(0.01, 1.00, 10)

# Toy response curves. These are simple saturating curves, not the full ODE model.
toy_p53_s15 = 0.75 * ddr_doses / (0.20 + ddr_doses)
toy_p53_s46 = 0.55 * ddr_doses**2 / (0.35**2 + ddr_doses**2)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8))

# Left panel: conceptual workflow.
axes[0].axis("off")
workflow_steps = [
    "DDR input",
    "p53 ODE model",
    "p53 S15/S46\nresponse scores",
    "Survival or\ndrug-response testing",
]
x_positions = [0.07, 0.33, 0.60, 0.88]

for x_position, label in zip(x_positions, workflow_steps):
    axes[0].text(
        x_position,
        0.55,
        label,
        ha="center",
        va="center",
        fontsize=9.5,
        bbox={"boxstyle": "round,pad=0.30", "facecolor": "white", "edgecolor": "black"},
    )

for start, end in zip(x_positions[:-1], x_positions[1:]):
    axes[0].annotate(
        "",
        xy=(end - 0.105, 0.55),
        xytext=(start + 0.105, 0.55),
        arrowprops={"arrowstyle": "->", "linewidth": 1.5},
    )

axes[0].set_title("Conceptual workflow", fontsize=12, weight="bold")

# Right panel: toy DDR response curves.
axes[1].plot(ddr_doses, toy_p53_s15, marker="o", label="Toy p53 S15 score")
axes[1].plot(ddr_doses, toy_p53_s46, marker="s", label="Toy p53 S46 score")
axes[1].set_title("Toy DDR dose-response", fontsize=12, weight="bold")
axes[1].set_xlabel("DDR input")
axes[1].set_ylabel("Simulated response score")
axes[1].set_ylim(0, 0.85)
axes[1].grid(alpha=0.3)
axes[1].legend(frameon=False)

fig.suptitle("p53 ODE workflow for the assignment", fontsize=14, weight="bold")
fig.tight_layout()

figure_path = figures_dir / "p53_ode_conceptual_workflow.png"
fig.savefig(figure_path, dpi=200, bbox_inches="tight")

figure_path

## Interpretation of the Toy Plot

The toy plot shows the shape of a possible model workflow, not real biology.

In the full workflow, each TCGA-BRCA sample would have molecular values for the p53 model input genes. The model would then simulate p53 S15 and p53 S46 response scores across DDR inputs. Those scores would become features for later survival or drug-response testing.

The key assignment distinction is that these scores come from a pre-specified mechanistic model. They are not learned directly from TCGA-BRCA survival labels.

## Next Steps

- Prepare TCGA-BRCA expression and clinical data in `00_data_preparation.ipynb`.
- Decide how to map TCGA-BRCA expression values onto the p53 model input genes.
- Port the professor p53 equations carefully once the data inputs are ready.
- Keep the model readable and explain each equation in assignment language.
- Use later notebooks for survival analysis, cell-line drug-response modelling, transfer analysis, and LINCS signatures.